# Разметка ЭКГ эксперимента 2

**Статус:** активный производитель кандидатов R-зубцов и модельных окон
электрической систолы. Автоматическая детекция требует ручного контроля.

Ноутбук выделен из исторического 08_Скетч_разметки.ipynb. Он не определяет
механические или клапанные события по боковой реограмме.


## Входы и научный статус

Входами служат исходный CSV и дыхательный sidecar той же записи из 11.01.
Связь проверяется по SHA-256. Частота дискретизации оценивается по медиане
разностей TIME_s; это расчётная величина, а не паспортная частота прибора.

R-зубцы являются результатом алгоритмической детекции. Интервал от принятого
начала Q до расчётного конца T является модельным окном. Он не заменяет
измеренную QT-разметку, механическую систолу или моменты открытия и закрытия
клапанов.


In [ ]:
# Импорты и внешняя конфигурация
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks

CONFIG_ENV = "KALMYKOV_EXP02_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp02_paths.example.json"
    )

CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
BREATHING_DIR = DERIVED_ROOT / "exp02" / "annotations" / "breathing"
OUT_DIR = DERIVED_ROOT / "exp02" / "annotations" / "ecg"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "TIME_s"
ECG_COL = "ECG_V"
ALGORITHM_VERSION = "exp02-ecg-rpeak-v1"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def sampling_frequency(df):
    time = df[TIME_COL].to_numpy(dtype=float)
    delta = np.diff(time)
    if len(delta) == 0 or np.any(~np.isfinite(delta)) or np.any(delta <= 0):
        raise ValueError("TIME_s должен быть конечным и строго возрастающим")
    median_dt = float(np.median(delta))
    jitter_fraction = float(
        np.median(np.abs(delta - median_dt)) / median_dt
    )
    return 1.0 / median_dt, jitter_fraction


In [ ]:
# Детекция R-зубцов и модельное окно электрической систолы
def detect_rpeaks(ecg, fs_hz):
    high_hz = min(15.0, fs_hz / 2.0 - 1.0)
    if high_hz <= 5.0:
        raise ValueError("Частота дискретизации недостаточна для полосы 5–15 Гц")

    b, a = butter(
        2,
        [5.0 / (fs_hz / 2.0), high_hz / (fs_hz / 2.0)],
        btype="band",
    )
    filtered = filtfilt(b, a, np.asarray(ecg, dtype=float))
    filtered /= np.std(filtered) + 1e-9
    candidates, _ = find_peaks(
        filtered,
        distance=max(1, int(0.33 * fs_hz)),
        height=2.0,
        prominence=1.5,
    )

    refined = []
    half_window = max(1, int(0.05 * fs_hz))
    raw = np.asarray(ecg, dtype=float)
    for candidate in candidates:
        start = max(0, candidate - half_window)
        stop = min(len(raw), candidate + half_window)
        refined.append(start + int(np.argmax(raw[start:stop])))
    return np.asarray(sorted(set(refined)), dtype=int)

def model_electrical_systole(rpeaks_s, qtc_s=0.40, q_lead_s=0.040):
    windows = []
    rpeaks_s = np.asarray(rpeaks_s, dtype=float)
    for index, r_time in enumerate(rpeaks_s):
        if index + 1 < len(rpeaks_s):
            rr_s = rpeaks_s[index + 1] - r_time
        elif index > 0:
            rr_s = r_time - rpeaks_s[index - 1]
        else:
            rr_s = 0.8

        qt_s = qtc_s * np.sqrt(max(rr_s, 0.3))
        q_start_s = r_time - q_lead_s
        t_end_s = q_start_s + qt_s
        windows.append([float(q_start_s), float(t_end_s)])
    return windows


In [ ]:
# Построение отдельных ЭКГ-sidecar-файлов
records = []
for breathing_path in sorted(BREATHING_DIR.glob("*.json")):
    breathing = json.loads(breathing_path.read_text(encoding="utf-8"))
    source_path = DATA_ROOT / breathing["input"]["relative_path"]
    input_sha256 = sha256_file(source_path)
    if input_sha256 != breathing["input"]["sha256"]:
        raise RuntimeError(
            f"Исходный файл изменился после дыхательной разметки: "
            f"{breathing['record_id']}"
        )

    frame = pd.read_csv(source_path, encoding="utf-8")
    fs_hz, jitter_fraction = sampling_frequency(frame)
    peak_indices = detect_rpeaks(frame[ECG_COL].to_numpy(dtype=float), fs_hz)
    time = frame[TIME_COL].to_numpy(dtype=float)
    rpeaks_s = [float(time[index]) for index in peak_indices]
    windows = model_electrical_systole(rpeaks_s)

    output = {
        "schema_version": 1,
        "annotation_type": "ecg",
        "algorithm_version": ALGORITHM_VERSION,
        "record_id": breathing["record_id"],
        "subject_id": breathing["subject_id"],
        "size_mm": breathing["size_mm"],
        "input": {
            "relative_path": breathing["input"]["relative_path"],
            "sha256": input_sha256,
            "sampling_frequency_hz": fs_hz,
            "sampling_frequency_source": "median_diff_TIME_s",
            "relative_time_step_jitter": jitter_fraction,
        },
        "rpeaks_s": rpeaks_s,
        "model_electrical_systole_s": windows,
        "model_parameters": {
            "qtc_s": 0.40,
            "q_lead_s": 0.040,
            "qt_definition": "q_start_to_t_end",
        },
        "upstream_breathing_qc": breathing["qc"]["status"],
        "qc": {
            "status": "pending_manual_review",
            "reviewer": None,
            "reviewed_at": None,
            "notes": None,
        },
    }
    output_path = OUT_DIR / f"{breathing['record_id']}.json"
    output_path.write_text(
        json.dumps(output, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    records.append(output)
    print(output["record_id"], len(rpeaks_s), output["qc"]["status"])

print("Создано кандидатов:", len(records))
print("Каталог:", OUT_DIR)


## Ручной контроль качества

Для каждой записи необходимо проверить пропущенные и ложные R-зубцы, участки
с артефактами, стабильность полярности и пригодность модельного окна. Статус
дыхательной разметки переносится только как информация о состоянии и не
принимает ЭКГ-разметку автоматически.


In [ ]:
# Контрольный график выбранного кандидата без автоматического принятия
CHECK_RECORD_ID = None
WINDOW_S = None

if CHECK_RECORD_ID is None:
    print("Задайте CHECK_RECORD_ID после построения кандидатов.")
else:
    sidecar_path = OUT_DIR / f"{CHECK_RECORD_ID}.json"
    annotation = json.loads(sidecar_path.read_text(encoding="utf-8"))
    source_path = DATA_ROOT / annotation["input"]["relative_path"]
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("Хеш исходного CSV изменился после разметки")

    frame = pd.read_csv(source_path, encoding="utf-8")
    time = frame[TIME_COL].to_numpy(dtype=float)
    ecg = frame[ECG_COL].to_numpy(dtype=float)
    mask = np.ones(len(time), dtype=bool)
    if WINDOW_S is not None:
        mask = (time >= WINDOW_S[0]) & (time <= WINDOW_S[1])

    fig, axis = plt.subplots(figsize=(14, 4))
    axis.plot(time[mask], ecg[mask], color="0.35", linewidth=0.7)
    visible_time = time[mask]
    for r_time in annotation["rpeaks_s"]:
        if len(visible_time) and visible_time[0] <= r_time <= visible_time[-1]:
            axis.axvline(r_time, color="red", linewidth=0.6)
    axis.set_xlabel("Время, с")
    axis.set_ylabel("ЭКГ, В")
    axis.set_title(
        f"{CHECK_RECORD_ID}: кандидаты R-зубцов; "
        f"QC={annotation['qc']['status']}"
    )
    plt.tight_layout()
    plt.show()


## Выход и зависимые этапы

Выход находится во внешнем каталоге derived/exp02/annotations/ecg/.
Дыхательная и ЭКГ-разметка связываются по record_id и SHA-256 исходного CSV,
но имеют независимые версии алгоритма и статусы контроля качества.
